# Baseline Book of Life Texts for Temporal Delinquency Targets

This notebook tests the Book-of-Life idea using the temporally defined target setup.

The design is intentionally stricter than the previous broad persistent BoL draft:

- **Predictors:** only stable background variables and survey information observed before 2000
- **Target:** delinquency/contact outcomes observed between 2000 and 2020
- **Main target:** `later_persistent_delinquency_contact_2000_2020`

The goal is to avoid using information from the outcome window as predictors.

## Load Data

The notebook uses three inputs:

1. `nlsy79_child_youngadult_selected_crime_features.csv` from the project root
2. the temporal targets created in `BoL approach 2/data/targets/`
3. the existing broad BoL feature index from the draft as metadata, then filtered to pre-2000 features

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "BoL approach 2":
    APPROACH_DIR = cwd.parent
elif cwd.name == "BoL approach 2":
    APPROACH_DIR = cwd
else:
    APPROACH_DIR = cwd / "BoL approach 2"
PROJECT_DIR = APPROACH_DIR.parent
TARGET_DIR = APPROACH_DIR / "data" / "targets"
FEATURE_DIR = APPROACH_DIR / "data" / "features"
BOOK_DIR = APPROACH_DIR / "data" / "books"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
BOOK_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
TARGETS_PATH = TARGET_DIR / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
FEATURE_META_PATH = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"

OUT_FEATURE_INDEX = FEATURE_DIR / "baseline_pre2000_bol_feature_index.csv"
OUT_BOOKS_JSON = BOOK_DIR / "baseline_pre2000_bol_books_sample_120.json"
OUT_SAMPLE_IDS = FEATURE_DIR / "baseline_pre2000_bol_sample_ids_120.csv"

TARGET = "later_persistent_delinquency_contact_2000_2020"
SAMPLE_N = 120
MAX_FEATURES_PER_GROUP_YEAR = 30

MISSING_CODES = {-1, -2, -3, -4, -5, -7}
RACE_LABELS = {1: "Hispanic", 2: "Black", 3: "Non-Black, non-Hispanic"}
SEX_LABELS = {1: "male", 2: "female"}

GROUP_ORDER = [
    "background",
    "family/home",
    "socioeconomic/work",
    "school/achievement",
    "behavioral/externalizing",
    "peer/neighborhood",
    "substance/risk behavior",
    "prior delinquency/contact",
    "health",
    "other",
]

data = pd.read_csv(DATA_PATH)
targets = pd.read_csv(TARGETS_PATH)
feature_meta = pd.read_csv(FEATURE_META_PATH)

data.shape, targets.shape, feature_meta.shape

ModuleNotFoundError: No module named 'numpy'

## Build a Strict Baseline Feature Index

The previous broad BoL draft included features through 2020 and relied on person-specific cutoffs. Here we want a clean baseline prediction setup. Therefore we keep only:

- `XRND` stable background variables
- survey-year features with `survey_year < 2000`
- features that are actually present in the feature CSV

We exclude ID variables and any target-style columns.

In [ ]:
feature_meta = feature_meta.copy()
feature_meta["year_num"] = pd.to_numeric(feature_meta["survey_year"], errors="coerce")

excluded_codes = {
    "C0000100", "C0000200",
    "target_property_ever", "target_property_first_event_year",
    "target_violent_ever", "target_violent_first_event_year",
}

in_data = feature_meta["csv_code"].isin(data.columns)
is_static = feature_meta["survey_year"].eq("XRND")
is_pre2000 = feature_meta["year_num"].lt(2000)
not_excluded = ~feature_meta["csv_code"].isin(excluded_codes)
not_constructed = ~feature_meta["selection_source"].astype(str).str.contains("constructed_target", na=False)

feature_index = feature_meta.loc[
    in_data & (is_static | is_pre2000) & not_excluded & not_constructed
].copy()

feature_index = feature_index.sort_values(["survey_year", "feature_group", "topic", "csv_code"])
feature_index.to_csv(OUT_FEATURE_INDEX, index=False)

print("Baseline feature count:", len(feature_index))
print("Unique csv_code features:", feature_index["csv_code"].nunique())
print("Static features:", int(is_static.loc[feature_index.index].sum()))
print("Pre-2000 year-specific features:", int(is_pre2000.loc[feature_index.index].sum()))
print("Saved feature index to:", OUT_FEATURE_INDEX)

## Feature Analysis

The current baseline index counts features at the year-specific `csv_code` level. If the same question appears across multiple survey years, it is counted multiple times. The collapsed question-level count shows how many distinct question texts remain after ignoring repeated survey waves.

In [ ]:
feature_index["question_norm"] = (
    feature_index["question"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

feature_summary = (
    feature_index
    .groupby("feature_group", dropna=False)
    .agg(
        year_specific_features=("csv_code", "nunique"),
        question_level_features=("question_norm", "nunique"),
    )
    .reset_index()
)
feature_summary["difference"] = feature_summary["year_specific_features"] - feature_summary["question_level_features"]
feature_summary = feature_summary.sort_values("year_specific_features", ascending=False)
feature_summary

In [ ]:
year_summary = (
    feature_index
    .assign(display_year=lambda d: d["survey_year"].astype(str))
    .groupby("display_year")
    .agg(n_features=("csv_code", "nunique"))
    .reset_index()
    .sort_values("display_year")
)
year_summary

In [ ]:
feature_index[["csv_code", "ref_id", "variable", "survey_year", "topic", "feature_group", "selection_source", "question"]].head(20)

## Target Balance for the Main Outcome

The sample for text generation is balanced on the main persistent delinquency/contact target, using only respondents with non-missing target values.

In [ ]:
target_counts = targets[TARGET].value_counts(dropna=False).rename_axis(TARGET).reset_index(name="n")
target_counts["share"] = target_counts["n"] / len(targets)
target_counts

In [ ]:
eligible = targets[targets[TARGET].notna()].copy()
base_rate = eligible[TARGET].mean()
print("Eligible rows:", len(eligible))
print("Base rate:", round(base_rate, 3))
print("Positive:", int((eligible[TARGET] == 1).sum()))
print("Negative:", int((eligible[TARGET] == 0).sum()))

## Render Feature Values as Book-of-Life Text

Each selected variable is converted into a short sentence using the question text from the feature index. The final book is organized as:

- stable background information
- year-by-year sections before 2000
- feature groups within each year

In [ ]:
def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)) and int(value) in MISSING_CODES:
        return None
    if isinstance(value, str) and value.strip() in {str(x) for x in MISSING_CODES}:
        return None
    return value


def safe_int(value):
    try:
        if pd.isna(value):
            return None
        return int(float(value))
    except (TypeError, ValueError):
        return None


def clean_question(question):
    question = str(question or "").strip()
    question = " ".join(question.split()).replace("#", "number of")
    if not question:
        return "value"
    letters = [ch for ch in question if ch.isalpha()]
    if sum(ch.isupper() for ch in letters) / max(1, len(letters)) > 0.7:
        question = question.lower()
    return question[:1].upper() + question[1:]


def render_value(code, value):
    value = clean_value(value)
    if value is None:
        return None
    as_int = safe_int(value)
    if code == "C0005300" and as_int in RACE_LABELS:
        return RACE_LABELS[as_int]
    if code == "C0005400" and as_int in SEX_LABELS:
        return SEX_LABELS[as_int]
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def sentence_for(code, value, meta_by_code):
    raw_value = clean_value(value)
    if raw_value is None:
        return None
    meta = meta_by_code.get(code, {})
    question = clean_question(meta.get("question", meta.get("variable", code)))
    as_int = safe_int(raw_value)
    question_upper = question.upper()
    yes_no_like = (
        as_int in (0, 1)
        and (
            question_upper.startswith(("HAS ", "HAVE ", "HAD ", "EVER ", "WAS ", "IS ", "DOES ", "DID "))
            or " EVER " in question_upper
            or " IN PAST " in question_upper
            or " IN LAST YEAR " in question_upper
        )
    )
    rendered = "yes" if yes_no_like and as_int == 1 else "no" if yes_no_like and as_int == 0 else render_value(code, raw_value)
    if rendered is None:
        return None
    return f"{question}: {rendered}"

In [ ]:
def book_for_child(row, features, meta_by_code):
    child_id = int(row["C0000100"])
    mother_id = int(row["C0000200"]) if not pd.isna(row.get("C0000200")) else "unknown"
    lines = [
        f"This Book of Life describes NLSY79 child respondent {child_id}, linked to mother ID {mother_id}.",
        "Only stable background variables and information observed before 2000 are included.",
    ]

    static = features[features["survey_year"].eq("XRND")]
    static_parts = []
    for code in static["csv_code"]:
        sentence = sentence_for(code, row.get(code), meta_by_code)
        if sentence:
            static_parts.append(sentence)
    if static_parts:
        lines.append("Stable background information: " + "; ".join(static_parts) + ".")

    yearly = features[~features["survey_year"].eq("XRND")].copy()
    yearly["year"] = pd.to_numeric(yearly["survey_year"], errors="coerce")
    yearly = yearly[yearly["year"].notna() & (yearly["year"] < 2000)]
    yearly = yearly.sort_values(["year", "feature_group", "topic", "csv_code"])

    for year, group in yearly.groupby("year"):
        group_lines = []
        for feature_group in GROUP_ORDER:
            subgroup = group[group["feature_group"] == feature_group]
            if subgroup.empty:
                continue
            parts = []
            for code in subgroup["csv_code"]:
                sentence = sentence_for(code, row.get(code), meta_by_code)
                if sentence:
                    parts.append(sentence)
                if len(parts) >= MAX_FEATURES_PER_GROUP_YEAR:
                    break
            if parts:
                group_lines.append(f"{feature_group}: " + "; ".join(parts))
        if group_lines:
            lines.append(f"In {int(year)}: " + " | ".join(group_lines) + ".")

    return "\n".join(lines)

## Generate a Balanced 120-Person Sample

This mirrors the draft's small sample idea: 60 positive and 60 negative cases for quick inspection and later LLM testing.

In [ ]:
if OUT_SAMPLE_IDS.exists():
    sample = pd.read_csv(OUT_SAMPLE_IDS)
else:
    pos = eligible[eligible[TARGET] == 1].sample(n=SAMPLE_N // 2, random_state=3026)
    neg = eligible[eligible[TARGET] == 0].sample(n=SAMPLE_N - len(pos), random_state=3027)
    sample = pd.concat([pos, neg], ignore_index=True).sample(frac=1, random_state=3028)
    sample.to_csv(OUT_SAMPLE_IDS, index=False)

sample[TARGET].value_counts(dropna=False)

In [ ]:
meta_by_code = feature_index.set_index("csv_code").to_dict(orient="index")
data_by_id = data.set_index("C0000100", drop=False)

books = {}
for _, target_row in sample.iterrows():
    child_id = int(target_row["C0000100"])
    row = data_by_id.loc[child_id]
    books[str(child_id)] = {
        "text": book_for_child(row, feature_index, meta_by_code),
        "source": "nlsy79_child_youngadult_selected_crime_features",
        "target": TARGET,
        "target_value": int(target_row[TARGET]),
        "predictor_window": "XRND and survey years before 2000",
        "target_window": "2000-2020",
    }

OUT_BOOKS_JSON.write_text(json.dumps(books, indent=2, sort_keys=True))
print("Books written:", len(books))
print("Saved to:", OUT_BOOKS_JSON)
print("Average book length:", int(np.mean([len(v["text"]) for v in books.values()])))

## Inspect One Rendered Book

In [ ]:
first_id = sorted(books)[0]
print("Child ID:", first_id)
print("Target value:", books[first_id]["target_value"])
print()
print(books[first_id]["text"][:2500])

## Outputs

This notebook writes:

- `baseline_pre2000_bol_feature_index.csv`
- `baseline_pre2000_bol_sample_ids_120.csv`
- `baseline_pre2000_bol_books_sample_120.json`

These outputs are the strict-baseline counterpart to the previous broad persistent BoL draft.